# Proyecto de Empresa Aliada: Segmentación Comercial con Clustering

## Análisis de Datos de Ventas para la Marca Vanish

En este proyecto se desarrolla un análisis de segmentación utilizando técnicas de **Machine Learning No Supervisado**, específicamente el algoritmo **K-Means Clustering**, con el objetivo de identificar patrones y segmentos relevantes dentro de los datos de ventas asociados a la marca Vanish.

El análisis de clustering permite agrupar productos o regiones que presentan comportamientos similares en términos de ventas, características de producto o desempeño comercial. Este tipo de metodología es ampliamente utilizada en analítica empresarial para comprender mejor el mercado y apoyar la toma de decisiones estratégicas.

A través de este análisis se busca identificar segmentos clave que permitan detectar oportunidades de mejora en la estrategia comercial de la marca.

---

## Objetivos del proyecto

Los principales objetivos de este análisis son:

- Explorar y comprender la estructura de los datos disponibles.
- Seleccionar variables relevantes para el proceso de segmentación.
- Aplicar técnicas de estandarización de datos.
- Implementar el algoritmo de **K-Means** para identificar grupos o clusters.
- Determinar el número óptimo de clusters utilizando el **método del codo**.
- Visualizar los resultados del clustering mediante gráficos.
- Analizar los clusters obtenidos para identificar **insights comerciales relevantes**.

---

## Flujo de trabajo del análisis

El desarrollo del proyecto seguirá las siguientes etapas:

1. Carga y exploración inicial de los datos
2. Preparación y limpieza de los datasets
3. Selección de variables para clustering
4. Estandarización de características
5. Aplicación del algoritmo K-Means
6. Determinación del número óptimo de clusters
7. Visualización de los segmentos identificados
8. Interpretación de resultados e insights de negocio

---

Este tipo de análisis permite comprender cómo se agrupan los productos o regiones según su comportamiento de ventas, facilitando la identificación de áreas donde la marca puede fortalecer su presencia o mejorar su estrategia comercial.

In [ ]:
# ============================================
# Importación de librerías
# ============================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# Configuración visual
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10,6)

## Carga de los datasets

En esta sección se cargan los diferentes archivos que conforman el modelo de datos del proyecto.  
Cada archivo contiene información relevante sobre distintos aspectos del negocio.

Los datasets incluidos son:

- **DIM_CALENDAR**: información temporal utilizada para análisis de ventas por fecha.
- **DIM_PRODUCT**: características y atributos de los productos.
- **DIM_SEGMENT**: segmentación de mercado o clasificación estratégica.
- **FACT_SALES**: tabla principal que contiene los registros de ventas.

La integración de estos datasets permitirá construir una vista analítica que será utilizada posteriormente para aplicar técnicas de clustering.

In [ ]:
# ============================================
# Carga de datasets
# ============================================

calendar = pd.read_excel("DIM_CALENDAR (2).xlsx")

product = pd.read_excel("DIM_PRODUCT (1).xlsx")

segment = pd.read_excel("DIM_SEGMENT (1).xlsx")

sales = pd.read_csv("FACT_SALES (1).csv")


# Mostrar primeras filas de cada dataset
print("Calendar")
display(calendar.head())

print("Product")
display(product.head())

print("Segment")
display(segment.head())

print("Sales")
display(sales.head())

## Exploración inicial de los datos

Antes de aplicar cualquier modelo de Machine Learning es fundamental comprender la estructura de los datos disponibles.

En esta etapa se revisan aspectos clave como:

- Número de registros
- Tipos de datos
- Variables disponibles
- Posibles valores faltantes

Este análisis exploratorio permite identificar qué variables serán útiles para el proceso de segmentación mediante clustering.

In [ ]:
# ============================================
# Información de los datasets
# ============================================

print("Calendar Dataset")
calendar.info()

print("\nProduct Dataset")
product.info()

print("\nSegment Dataset")
segment.info()

print("\nSales Dataset")
sales.info()

In [ ]:
# ============================================
# Revisar columnas de cada dataset
# ============================================

print("Calendar Columns")
print(calendar.columns)

print("\nProduct Columns")
print(product.columns)

print("\nSegment Columns")
print(segment.columns)

print("\nSales Columns")
print(sales.columns)

## Comprensión del modelo de datos

Antes de construir el dataset analítico es necesario comprender cómo se relacionan los distintos datasets disponibles.

El conjunto de datos sigue una estructura común en analítica empresarial conocida como **modelo estrella**, donde:

- La tabla **FACT_SALES** contiene las métricas principales del negocio (ventas).
- Las tablas **DIM_PRODUCT**, **DIM_SEGMENT** y **DIM_CALENDAR** contienen información descriptiva o dimensional que permite contextualizar dichas ventas.

Este modelo facilita la integración de datos para realizar análisis avanzados, como segmentación mediante algoritmos de clustering.

In [ ]:
# ============================================
# Tamaño de los datasets
# ============================================

print("Calendar shape:", calendar.shape)
print("Product shape:", product.shape)
print("Segment shape:", segment.shape)
print("Sales shape:", sales.shape)

In [ ]:
# ============================================
# Construcción del dataset analítico de ventas
# ============================================

sales_summary = sales.groupby("ITEM_CODE").agg({

    "TOTAL_UNIT_SALES": "sum",
    "TOTAL_VALUE_SALES": "sum",
    "TOTAL_UNIT_AVG_WEEKLY_SALES": "mean"

}).reset_index()

# Renombrar columnas para facilitar análisis
sales_summary.columns = [
    "item_code",
    "total_units",
    "total_revenue",
    "avg_weekly_units"
]

sales_summary.head()

## Construcción del dataset analítico

Para realizar el análisis de clustering es necesario construir un dataset que represente el comportamiento de ventas de los productos.

En esta etapa se agregan las ventas utilizando la variable **ITEM_CODE**, generando métricas clave como:

- Unidades totales vendidas
- Ingresos totales generados
- Promedio de ventas semanales

Estas métricas permiten capturar el desempeño comercial de cada producto y facilitan la aplicación de técnicas de segmentación mediante algoritmos de clustering.

In [ ]:
# ============================================
# Selección de variables para clustering
# ============================================

features = sales_summary[[
    
    "total_units",
    "total_revenue",
    "avg_weekly_units"

]]

features.head()

In [ ]:
# ============================================
# Distribución de variables
# ============================================

features.hist(bins=30)

plt.show()

## Estandarización de las variables

Antes de aplicar el algoritmo K-Means es necesario estandarizar las variables seleccionadas.  
Esto es importante porque K-Means calcula distancias entre los puntos de datos, por lo que las variables con escalas mayores podrían influir más en el resultado del modelo.

Para evitar este problema se utiliza **StandardScaler**, una técnica que transforma los datos para que todas las variables tengan una media cercana a 0 y una desviación estándar de 1.

Este proceso garantiza que todas las variables tengan la misma importancia dentro del algoritmo de clustering.

In [ ]:
# ============================================
# Estandarización de variables
# ============================================

scaler = StandardScaler()

scaled_features = scaler.fit_transform(features)

scaled_features[:5]

## Determinación del número óptimo de clusters

Para aplicar el algoritmo K-Means es necesario determinar cuántos clusters se deben crear.

Una técnica común para identificar el número óptimo de clusters es el **método del codo (Elbow Method)**.

Este método consiste en ejecutar el algoritmo K-Means con diferentes valores de K y calcular la suma de las distancias de los puntos a su centroide correspondiente.

El punto donde la reducción de la distancia comienza a disminuir de forma más lenta indica el número adecuado de clusters.

In [ ]:
# ============================================
# Método del Codo
# ============================================

inertia = []

k_range = range(1,10)

for k in k_range:
    
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    kmeans.fit(scaled_features)
    
    inertia.append(kmeans.inertia_)

plt.plot(k_range, inertia, marker='o')

plt.title("Elbow Method - Optimal Clusters")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")

plt.show()

## Aplicación del algoritmo K-Means

Una vez identificado el número óptimo de clusters mediante el método del codo, se aplica el algoritmo K-Means para segmentar los productos en diferentes grupos.

Cada cluster representará un conjunto de productos que comparten características similares en términos de comportamiento de ventas.

In [ ]:
# ============================================
# Aplicar K-Means
# ============================================

kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)

clusters = kmeans.fit_predict(scaled_features)

sales_summary["cluster"] = clusters

sales_summary.head()

In [ ]:
# ============================================
# Centroides del modelo KMeans
# ============================================

centroids = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=features.columns
)

centroids["cluster"] = range(len(centroids))

print("Centroides de los clusters:")
display(centroids)

In [ ]:
plt.figure(figsize=(10,6))

scatter = plt.scatter(
    sales_summary["total_units"],
    sales_summary["total_revenue"],
    c=sales_summary["cluster"],
    cmap="viridis",
    alpha=0.7
)

plt.xlabel("Total Units Sold")
plt.ylabel("Total Revenue")
plt.title("Product Clusters Based on Sales Performance")

plt.colorbar(scatter, label="Cluster")

plt.show()

## Visualización de clusters mediante PCA

Para visualizar mejor los clusters generados por el algoritmo K-Means, se utiliza la técnica de **Análisis de Componentes Principales (PCA)**.

PCA permite reducir la dimensionalidad de los datos transformando las variables originales en un número menor de componentes que conservan la mayor parte de la información del dataset.

En este caso, se reducen las variables a **dos componentes principales**, lo que permite representar gráficamente los clusters en un plano bidimensional.

In [ ]:
# ============================================
# Reducción de dimensionalidad con PCA
# ============================================

pca = PCA(n_components=2)

pca_features = pca.fit_transform(scaled_features)

pca_df = pd.DataFrame(
    pca_features,
    columns=["PCA1","PCA2"]
)

pca_df["cluster"] = clusters

pca_df.head()

In [ ]:
plt.figure(figsize=(10,6))

scatter = plt.scatter(
    pca_df["PCA1"],
    pca_df["PCA2"],
    c=pca_df["cluster"],
    cmap="viridis",
    alpha=0.7
)

plt.title("Cluster Visualization using PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.colorbar(scatter, label="Cluster")

plt.show()

## Interpretación de los clusters

Después de aplicar el algoritmo K-Means se identificaron diferentes grupos de productos que comparten características similares en términos de desempeño de ventas.

Cada cluster representa un segmento de productos con patrones de comportamiento específicos.

El análisis de estos segmentos permite identificar:

- Productos con **alto volumen de ventas**
- Productos con **ventas moderadas**
- Productos con **bajo desempeño comercial**

Este tipo de segmentación permite a las empresas identificar oportunidades estratégicas, como fortalecer la presencia de productos exitosos o revisar estrategias comerciales para productos con bajo rendimiento.

In [ ]:
# ============================================
# Análisis de clusters (solo variables numéricas)
# ============================================

sales_summary.groupby("cluster").mean(numeric_only=True)

In [ ]:
# ============================================
# Perfil detallado de cada cluster
# ============================================

cluster_profile = sales_summary.groupby("cluster").agg({

    "total_units":"mean",
    "total_revenue":"mean",
    "avg_weekly_units":"mean"

}).round(2)

print("Perfil promedio por cluster:")
display(cluster_profile)

In [ ]:
print("Cantidad de productos por cluster:")
display(sales_summary["cluster"].value_counts())

In [ ]:
# ============================================
# Distribución de productos por cluster
# ============================================

plt.figure(figsize=(8,5))

sns.countplot(
    x="cluster",
    data=sales_summary
)

plt.title("Distribución de productos por cluster")
plt.xlabel("Cluster")
plt.ylabel("Cantidad de productos")

plt.show()

In [ ]:
# ============================================
# Guardar resultados del clustering
# ============================================

sales_summary.to_csv("vanish_clusters_results.csv", index=False)

print("Archivo de resultados guardado correctamente")

## Insights estratégicos del análisis de clustering

A partir de la segmentación realizada con el algoritmo K-Means se identificaron diferentes patrones en el comportamiento de ventas de los productos. Estos resultados permiten generar conclusiones estratégicas que pueden ser utilizadas para mejorar la toma de decisiones comerciales.

### Insight 1: Identificación de productos de alto desempeño

Uno de los clusters agrupa productos con niveles significativamente altos de ventas totales y mayor frecuencia de transacciones. Este segmento representa productos con fuerte aceptación en el mercado.

Estos productos pueden considerarse **productos clave del portafolio**, por lo que la empresa podría priorizar su disponibilidad en inventario, fortalecer su distribución y apoyar su promoción comercial para maximizar su impacto en ingresos.

### Insight 2: Segmento de productos con desempeño moderado

Otro cluster agrupa productos con niveles intermedios de ventas y frecuencia de compra. Este segmento puede representar productos con demanda estable pero con potencial de crecimiento.

Para estos productos, estrategias como promociones específicas, mejoras en visibilidad en puntos de venta o campañas de marketing dirigidas podrían incrementar su desempeño comercial.

### Insight 3: Identificación de productos con bajo desempeño

Finalmente, se identificó un cluster compuesto por productos con niveles bajos de ventas y menor número de transacciones. Este grupo puede indicar productos con menor penetración en el mercado o menor reconocimiento por parte de los consumidores.

Este tipo de análisis permite a la empresa evaluar posibles acciones estratégicas como reposicionamiento del producto, ajustes en precio, mejoras en marketing o incluso la reconsideración de su permanencia dentro del portafolio.

En conjunto, el análisis de clustering permite comprender mejor el comportamiento del portafolio de productos y proporciona información valiosa para optimizar estrategias comerciales basadas en datos.

## Evaluación del modelo de clustering

Para evaluar la calidad de la segmentación generada por el algoritmo K-Means se utiliza el **Silhouette Score**.

Esta métrica permite medir qué tan bien se encuentran separados los clusters generados. Su valor se encuentra entre **-1 y 1**:

- Valores cercanos a **1** indican clusters bien separados
- Valores cercanos a **0** indican clusters solapados
- Valores negativos indican una mala asignación de clusters

Esta métrica proporciona una forma cuantitativa de evaluar la calidad del modelo de clustering.

In [ ]:
# ============================================
# Evaluación del clustering
# ============================================

score = silhouette_score(scaled_features, clusters)

print("Silhouette Score del modelo:", score)

## Interpretación del Silhouette Score

El modelo de clustering obtuvo un **Silhouette Score de 0.89**, lo que indica una segmentación de alta calidad.

Este valor sugiere que los clusters generados por el algoritmo K-Means presentan una **separación clara entre los grupos**, lo que significa que los productos dentro de cada cluster comparten características similares mientras que se diferencian significativamente de los productos en otros clusters.

Un Silhouette Score cercano a **1** indica que los datos están bien agrupados y que la asignación de clusters es adecuada para representar la estructura del dataset.

Este resultado confirma que el modelo de clustering logra identificar patrones relevantes dentro del comportamiento de ventas de los productos analizados.

In [ ]:
# ============================================
# Varianza explicada por PCA
# ============================================

explained_variance = pca.explained_variance_ratio_

print("Varianza explicada por cada componente:")
print("PCA1:", round(explained_variance[0],3))
print("PCA2:", round(explained_variance[1],3))

print("\nVarianza total explicada:", round(explained_variance.sum(),3))

In [ ]:
# ============================================
# Heatmap del perfil de clusters
# ============================================

plt.figure(figsize=(8,5))

sns.heatmap(
    cluster_profile,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Perfil promedio de cada cluster")

plt.show()

In [ ]:
# ============================================
# Asignar nombre estratégico a clusters
# ============================================

cluster_labels = {
    0: "Bajo desempeño",
    1: "Desempeño medio",
    2: "Alto desempeño"
}

sales_summary["cluster_name"] = sales_summary["cluster"].map(cluster_labels)

display(sales_summary.head())

## Conclusión

En este proyecto se aplicaron técnicas de aprendizaje automático no supervisado para analizar el comportamiento de ventas de productos asociados a la marca Vanish.

Mediante el uso del algoritmo K-Means fue posible identificar diferentes segmentos de productos con características similares en términos de ventas. La estandarización de variables y el uso de técnicas de reducción de dimensionalidad como PCA permitieron mejorar la interpretación visual de los clusters.

El análisis de estos segmentos puede ayudar a las empresas a comprender mejor el desempeño de sus productos en el mercado, identificar oportunidades de crecimiento y optimizar su estrategia comercial.

Este tipo de análisis demuestra el valor de la ciencia de datos para apoyar la toma de decisiones basada en datos dentro de las organizaciones.